### Datensatz Photolab Archives Albums
Import und öffnen des Browsers

In [172]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import re
import time
import os
import logging
from datetime import datetime
from bs4 import BeautifulSoup
from bs4 import NavigableString
from urllib.parse import urljoin
import csv

# 2. Logging setup (ONE TIME ONLY)
log_filename = f"scraper_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(log_filename, encoding="utf-8"),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger(__name__)


# -------------------------
# CONFIG
# -------------------------

CSV_FILE = "cern_photolab_records.csv"
PROGRESS_FILE = "progress_jrec.txt"

BASE_SEARCH_URL = (
    "https://cds.cern.ch/search?"
    "ln=en&cc=PhotoLab+Archives"
    "&rg=100"
    "&op1=a"
    "&m1=a"
)

# Start driver
driver = webdriver.Firefox()

# Open list page
driver.get("https://cds.cern.ch/collection/PhotoLab%20Archives?ln=en")

# Give page time to load
time.sleep(2)

### Load Progress

In [173]:
# -------------------------
# PROGRESS TRACKING (jrec)
# -------------------------

PROGRESS_FILE = "progress_jrec.txt"

def load_progress_jrec(default=1):
    """
    Returns the jrec value to start from.
    If progress file does not exist, start from default (1).
    """
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, "r", encoding="utf-8") as f:
            try:
                return int(f.read().strip())
            except:
                return default
    return default


def save_progress_jrec(jrec):
    """
    Saves the next jrec value so scraping can resume later.
    """
    with open(PROGRESS_FILE, "w", encoding="utf-8") as f:
        f.write(str(jrec))


### CSV Append

In [174]:
# -------------------------
# CSV APPEND (PER PAGE)
# -------------------------

CSV_FILE = "cern_photolab_records.csv"

CSV_HEADERS = [
    "Title",
    "Record URL",
    "Image URL",
    "Image File Name",
    "Row Number on Website",
    "Date",
    "Tirage",
    "Description"
]


def append_to_csv(rows):
    """
    Appends rows to CSV.
    Writes header only once.
    """
    file_exists = os.path.exists(CSV_FILE)

    with open(CSV_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        if not file_exists:
            writer.writerow(CSV_HEADERS)

        writer.writerows(rows)


### Initial search

In [ ]:

def initial_search_setup(driver, jrec):
    print(f"Running initial search setup (jrec={jrec})...")

   
    driver.get("https://cds.cern.ch/collection/PhotoLab%20Archives?ln=en")
    time.sleep(2)
    
    # Expand options
    more_link = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.LINK_TEXT, "[>> more]"))
    )
    more_link.click()

    # Select 100 results per page
    select_element = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.NAME, "rg"))
    )
    Select(select_element).select_by_value("100")

    # Click search
    search_button = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.NAME, "action_search"))
    )
    search_button.click()

    time.sleep(2)
    print("Initial search setup completed.")

    # Set jrec BEFORE search
    jrec_input = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "input[name='jrec']"))
    )
    jrec_input.clear()
    jrec_input.send_keys(str(jrec_value))

    # Click search 
    search_button = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.NAME, "action_search"))
    )
    search_button.click()

    # Wait for results
    time.sleep(2)

    print("Initial search setup completed.")


### add_picture_row_numbers

In [176]:
def add_picture_row_numbers(results):
    row_numbers = []
    for i, r in enumerate(results, start=1):
        img_td = r.find_all("td")[0]
        img = img_td.find("img")
        if img and img.get("src"):
            row_numbers.append(i)
    return row_numbers


### Scraper loop

In [177]:
def scrape_current_page(driver):
    titles = []
    links = []
    record_urls = []
    image_links = []
    image_filenames = []
    dates = []
    tirages = []
    descriptions = []
    output_rows = []

    
    soup = BeautifulSoup(driver.page_source, "html.parser")

    results = [
        tr for tr in soup.select("tbody > tr")
        if len(tr.find_all("td")) == 2
    ]

    print("Found rows:", len(results))

    # -------------------------
    # DATES
    # -------------------------
    date_pattern = re.compile(
        r"(?i)(\d{1,2}\s+[A-Za-z]{3,9}\s+\d{4})|"
        r"([A-Za-z]{3,9}\s+\d{4})|"
        r"(\d{4})|"
        r"(No date)"
    )

    for r in results:
        td = r.find_all("td")[1]
        date_text = None

        for br in td.find_all("br"):
            sib = br.next_sibling
            while sib and (not isinstance(sib, NavigableString) or not sib.strip()):
                sib = sib.next_sibling
            if not sib:
                continue

            text = sib.strip()
            if "[...]" in text:
                continue

            m = date_pattern.search(text)
            if m:
                date_text = m.group(0)
                break

        dates.append(date_text)

    # -------------------------
    # TIRAGES
    # -------------------------
    for r in results:
        td = r.find_all("td")[1]
        tirage_text = None

        em = td.find("em", string=lambda s: s and "Tirage" in s)
        if em:
            sib = em.next_sibling
            while sib and (not isinstance(sib, NavigableString) or not sib.strip()):
                sib = sib.next_sibling
            if sib:
                tirage_text = sib.strip()

        tirages.append(tirage_text)

    # -------------------------
    # TITLES + LINKS + IMAGES
    # -------------------------
    base_url = "https://cds.cern.ch"

    for r in results:
        cells = r.find_all("td")

        a = cells[1].find("a", class_="titlelink")
        if a:
            titles.append(a.get_text(strip=True))
            links.append(a.get("href"))
            record_urls.append(urljoin(base_url, a.get("href")))
        else:
            titles.append(None)
            links.append(None)
            record_urls.append(None)

        img = cells[0].find("img")
        if img and img.get("src"):
            img_url = urljoin(base_url, img.get("src"))
            image_links.append(img_url)

            if "/files/" in img_url:
                fname = img_url.split("/files/")[1].split(".jpg")[0]
            else:
                fname = None

            image_filenames.append(fname)
        else:
            image_links.append(None)
            image_filenames.append(None)

    # -------------------------
    # ROW NUMBERS
    # -------------------------
    row_numbers = add_picture_row_numbers(results)
    picture_row_index = 0

    # -------------------------
    # DESCRIPTIONS
    # -------------------------
    for record_url in record_urls:
        if not record_url:
            descriptions.append(None)
            continue

        driver.get(record_url)
        time.sleep(1)

        soup = BeautifulSoup(driver.page_source, "html.parser")
        desc_divs = soup.select("div.album-description")

        parts = []
        for div in desc_divs:
            for p in div.find_all("p"):
                t = p.get_text(strip=True)
                if t:
                    parts.append(t)

        descriptions.append(" ".join(parts) if parts else None)

        
    # -------------------------
    # BUILD OUTPUT ROWS
    # -------------------------
    for i in range(len(titles)):
        if not links[i] or not image_links[i]:
            continue

        if picture_row_index >= len(row_numbers):
            continue

        website_row_number = row_numbers[picture_row_index]
        picture_row_index += 1

        output_rows.append([
            titles[i],
            record_urls[i],
            image_links[i],
            image_filenames[i],
            website_row_number,
            dates[i],
            tirages[i],
            descriptions[i]
        ])

    return output_rows


### Main Loop

In [ ]:
# driver = webdriver.Firefox()

# # Always start from the first page
# initial_search_setup(driver, jrec)

page_counter = load_progress_jrec(default=1)
jrec_value = 1  # Start at record 1
max_records = 20708  # Total records to scrape

while True:
    print(f"Scraping page {page_counter}")

    rows = scrape_current_page(driver)
    logger.info(f"Extracted {len(rows)} rows")

    append_to_csv(rows)
    logger.info("CSV updated")

    # NEW: Navigate back to the search results page
    driver.back()  # Go back from the last record page
    time.sleep(2)  # Give it time to load

    jrec = 1 + (page_counter - 1) * 100
    initial_search_setup(driver, jrec)

  
    # Calculate next jrec (100 records per page)
    jrec_value += 100
    page_counter += 1
    save_progress_jrec(page_counter)
    # logger.info(f"Progress saved: page={page_counter}"), next jrec={jrec_value}")

     # If we haven't reached the end, navigate to next page
    if jrec_value <= max_records:
        initial_search_setup(driver, jrec_value)
    else:
        logger.info("Reached end of records")
        break

driver.quit()


Scraping page 1
Found rows: 13


2026-01-07 17:02:46,540 | INFO | Extracted 10 rows
2026-01-07 17:02:46,543 | INFO | CSV updated


NameError: name 'jrec' is not defined